In [99]:
import numpy as np
import sys
import os
from math import ceil, log10

# Thêm thư mục cha (L02) vào sys.path
parent_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if parent_path not in sys.path:
    sys.path.append(parent_path)

from src.data_processing import load_csv#, one_hot_encode

Đọc dữ liệu gốc từ file data/raw/BankChurner.csv

In [100]:
DATA_PATH = "../data/raw/BankChurners.csv"
OUT_DIR = "../data/processed"
data, header = load_csv(DATA_PATH)

Loại bỏ các dấu nháy kép ( " )

In [101]:
def strip_quotes(s):
    return s.strip().strip('"').strip() if isinstance(s, str) else s

v_strip = np.vectorize(strip_quotes)
header = list(v_strip(header))
data = v_strip(data)

Ép kiểu các giá trị số (numeric value)

In [102]:
def convert_numeric(x):
    if not isinstance(x, str):
        return x
    
    try:
        # 1. Thử ép kiểu INT (số nguyên)
        # Không phải '1.335' hay '9.3e-05'
        if '.' not in x and 'e' not in x.lower():
            return int(x)
    except (ValueError, TypeError):
        pass 
    
    try:
        # 2. Thử ép kiểu FLOAT (số thực)
        # Ép kiểu '1.335' và '9.3448e-05'
        return float(x)
    except (ValueError, TypeError):
        # 3. Nếu không phải dạng số
        # Trả về chuỗi gốc
        return x

# test = data[:,0].astype(int)
v_convert_numeric = np.vectorize(convert_numeric, otypes=[object])
data = v_convert_numeric(data)

Xử lý mã hóa các dữ liệu phân loại (feature encoding)

In [103]:
cat_col_names = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Attrition_Flag']
cat_col_idx = [header.index(c) for c in cat_col_names]

# Attrition_Flag
Attrition_idx = header.index('Attrition_Flag')
feat_attrition = np.where(data[:, Attrition_idx] == 'Attrited Customer', 1, 0).reshape(-1, 1)

# Gender
Gender_idx = header.index('Gender')
feat_gender = np.where(data[:, Gender_idx] == 'M', 1, 0).reshape(-1, 1)

def ordinal_encoding(col_idx, col_dict, col):
    col_keys = np.array(list(col_dict.keys()))
    col_values = np.array(list(col_dict.values()))
    sort_idx = np.argsort(col_keys)
    mapped = col_values[sort_idx]
    col_uni, inv = np.unique(col, return_inverse=True)
    return mapped[inv] # fancy indexing

# Education level (Ordinal Encoding)
# Từ điển lưu giá trị số sẽ mã hóa
Edu_idx = header.index('Education_Level')
Edu_dict = {'Unknown': 0, 'Uneducated': 1, 'High School': 2, 'College': 3, 'Graduate': 4, 'Post-Graduate': 5, 'Doctorate': 6}
feat_edu = ordinal_encoding(Edu_idx, Edu_dict, data[:, Edu_idx]).reshape(-1, 1)

# Income Category (Ordinal Encoding)
inc_idx = header.index('Income_Category')
inc_dict = {'Unknown': 0, 'Less than $40K': 1, '$40K - $60K': 2, '$60K - $80K': 3, '$80K - $120K': 4, '$120K +': 5}
feat_inc = ordinal_encoding(inc_idx, inc_dict, data[:, inc_idx]).reshape(-1, 1)

def one_hot_encoding(col):
    unique_vals = np.unique(col)
    
    col_reshaped = col.reshape(-1, 1) # shape (n, 1)
    # Dùng broadcasting để so sánh (n, 1) với (k,)
    # NumPy sẽ so sánh mỗi phần tử trong 'col_reshaped'
    # với TẤT CẢ các phần tử trong 'unique_vals'.
    # Kết quả là một ma trận Boolean (True/False) kích thước (n, k)
    boolean_matrix = (col_reshaped == unique_vals)
    
    return boolean_matrix.astype(int), unique_vals

# Marital Status (one hot encoding)
marital_idx = header.index('Marital_Status')
feat_marital, marital_uni = one_hot_encoding(data[:, marital_idx])

# Card Category (one hot encoding)
card_idx = header.index('Card_Category')
feat_card, card_uni = one_hot_encoding(data[:, card_idx])

data_cat_encoded = np.hstack((feat_gender, feat_edu, feat_inc, feat_marital, feat_card))
data_cat_encoded[:10]

array([[1, 2, 3, 0, 1, 0, 0, 1, 0, 0, 0],
       [0, 4, 1, 0, 0, 1, 0, 1, 0, 0, 0],
       [1, 4, 4, 0, 1, 0, 0, 1, 0, 0, 0],
       [0, 2, 1, 0, 0, 0, 1, 1, 0, 0, 0],
       [1, 1, 3, 0, 1, 0, 0, 1, 0, 0, 0],
       [1, 4, 2, 0, 1, 0, 0, 1, 0, 0, 0],
       [1, 0, 5, 0, 1, 0, 0, 0, 1, 0, 0],
       [1, 2, 3, 0, 0, 0, 1, 0, 0, 0, 1],
       [1, 1, 3, 0, 0, 1, 0, 1, 0, 0, 0],
       [1, 4, 4, 0, 0, 1, 0, 1, 0, 0, 0]])

Xử lý các giá trị ngoại lai (outliers) trên cột dữ liệu số

In [104]:
num_col_names = [
    'Customer_Age', 'Dependent_count', 'Months_on_book', 
    'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 
    'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 
    'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'
]
num_col_idx = [header.index(c) for c in num_col_names] # list

# Xử lí các giá trị ngoại lai (outliers) trên các cột có giá trị là số
data_num = data[:, num_col_idx]
Q1 = np.percentile(data_num, 25, axis=0)
Q3 = np.percentile(data_num, 75, axis=0)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
data_num_clipped = np.clip(data_num, lower_bound, upper_bound)

Thêm các đặc trưng dữ liệu (Feature Engineering)

In [ ]:
# Feature Engineering
num_header_list = list(num_col_names)
idx_Total_Trans_Amt = num_header_list.index('Total_Trans_Amt')
idx_Total_Trans_Ct = num_header_list.index('Total_Trans_Ct')
idx_Months_on_book = num_header_list.index('Months_on_book')
safe_ct = data_num_clipped[:, idx_Total_Trans_Ct] + 1e-6
safe_months = data_num_clipped[:, idx_Months_on_book] + 1e-6
engineered_headers = ['Avg_Trans_Value', 'Activity_Ratio']
# Đặc trưng tính giá trị giao dịch trung bình
feat_avg_trans_val = (data_num_clipped[:, idx_Total_Trans_Amt] / safe_ct).reshape(-1, 1)
# Đặc trưng tần suất hoạt động
feat_activity_ratio = (data_num_clipped[:, idx_Total_Trans_Ct] / safe_months).reshape(-1, 1)
data_num_engineered = np.hstack((data_num_clipped, feat_avg_trans_val, feat_activity_ratio))
data_num_engineered

array([[45, 3, 39, ..., 0.061, 27.23809458956918, 1.0769230493096655],
       [49, 5, 44, ..., 0.105, 39.12121093572088, 0.7499999829545458],
       [51, 3, 36, ..., 0, 94.34999528250023, 0.5555555401234572],
       ...,
       [44, 1, 36, ..., 0, 143.6541642724306, 1.6666666203703717],
       [30, 2, 36, ..., 0, 135.40322362252866, 1.7222221743827175],
       [43, 2, 25, ..., 0.189, 141.2991780114889, 2.439999902400004]],
      dtype=object)

Điều chuẩn Z-score cho dữ liệu đã mã hóa

In [106]:
X_final = np.hstack((data_cat_encoded, data_num_engineered))
X_final = X_final.astype(float)
mean = np.mean(X_final, axis=0)
std = np.std(X_final, axis=0)
safe_std = std + 1e-6
X_scaled = (X_final - mean) / safe_std
X_scaled

array([[ 1.05995352, -0.35402182,  0.62003849, ..., -0.77587942,
        -1.81037902, -0.95760241],
       [-0.94343381,  0.82221799, -0.73629152, ..., -0.61627342,
        -1.12064144, -1.33318114],
       [ 1.05995352,  0.82221799,  1.29820349, ..., -0.99715138,
         2.08503009, -1.55656457],
       ...,
       [-0.94343381, -0.35402182, -0.73629152, ..., -0.99715138,
         4.94681626, -0.28008784],
       [ 1.05995352,  0.82221799, -0.05812652, ..., -0.99715138,
         4.46790285, -0.216264  ],
       [-0.94343381,  0.82221799, -0.73629152, ..., -0.31157105,
         4.81012463,  0.60833993]])

Ghi dữ liệu đã xử lí ra thư mục data/processed

In [110]:
final_header_list = (
    cat_col_names[:1] +  
    cat_col_names[1:2] +      # OE
    cat_col_names[2:3] +      # OE
    list(marital_uni) +      # Marital (OHE)
    list(card_uni) +         # Card (OHE)
    num_col_names +          # 14 cột số
    engineered_headers +     # 2 cột mới (feature engineering)
    cat_col_names[-1:]       # Cột 'Attrition_Flag'
)
header_string = ','.join(final_header_list)

combined_data = np.hstack((X_scaled, feat_attrition))

out_dir = "../data/processed/BankChurners.csv"
out_path = os.path.dirname(out_dir)
os.makedirs(out_path, exist_ok=True)

np.savetxt(
    out_dir, 
    combined_data, 
    delimiter=',', 
    fmt='%.8f',
    header=header_string,  
    comments=''           
)